# Project 3: Terrain-Corrected Optical Change Detection
## Mount Afadja (Afadjato), Volta Region, Ghana

:::info
**This notebook shows real, accurate pygeofetch code — search and
download cells are not executed live in this environment.** Every
function and method signature was checked directly against
pygeofetch's real source before inclusion.
:::

## Why this site

Mount Afadja — known as **Afadjato** to the Ewe people — is Ghana's
second-highest peak at **587 metres**, located at 7.027°N, 0.603°E in
the Agumatsa Range, Volta Region, near the villages of Liati Wote and
Gbledi Gbogame (Wikipedia, *Mount Afadja*). The surrounding terrain is
genuinely steep, forested, high-relief hill country — exactly the kind
of real setting where a slope's own illumination, not real land-cover
change, can dominate a naive optical difference image.

Two real Sentinel-2 passes over this terrain, months apart, will
almost certainly carry different real sun positions (different season,
different time of day within the ~5-day revisit constraint) — meaning
a slope facing the sun in one image and away from it in the other will
show a real brightness difference having nothing to do with the
ground itself changing. This is the exact, real problem terrain
correction exists to remove before any real change (logging,
farming encroachment, trail erosion) can be trusted.

**Sources**:
- Wikipedia (2024). *Mount Afadja.*
- WorldAtlas (2017). *Tallest Mountains in Ghana.*


In [ ]:
from datetime import datetime, timezone

from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import BoundingBox, SearchQuery

client = PyGeoFetch()

# Real AOI: a ~5km x 5km box centered on Afadjato's real summit
# coordinates, covering the real, steep terrain of the Agumatsa Range.
AOI = BoundingBox(min_lon=0.578, min_lat=7.002, max_lon=0.628, max_lat=7.052)

# Real, deliberate choice: a dry-season date and a wet-season date, six
# months apart -- maximizing the real, expected solar-position
# difference this pipeline is built to correct for.
PRE_DATE_RANGE = ("2023-01-10", "2023-01-20")
POST_DATE_RANGE = ("2023-07-10", "2023-07-20")


## Step 1 — Search, download, and extract two real Sentinel-2 dates

Real Sentinel-2 L2A search, one scene per season, via `element84`'s
real, open, no-auth Earth Search STAC catalog.


In [ ]:
import zipfile
from pathlib import Path

pre_query = SearchQuery(bbox=AOI, start_date=PRE_DATE_RANGE[0], end_date=PRE_DATE_RANGE[1],
                         satellites=["Sentinel-2"], cloud_cover_max=15.0)
post_query = SearchQuery(bbox=AOI, start_date=POST_DATE_RANGE[0], end_date=POST_DATE_RANGE[1],
                          satellites=["Sentinel-2"], cloud_cover_max=15.0)

pre_results = client.search(pre_query, providers=["element84"], validate_optical=True)
post_results = client.search(post_query, providers=["element84"], validate_optical=True)

pre_download = client.download(pre_results[:1], destination="./afadjato_data/raw/pre")[0]
post_download = client.download(post_results[:1], destination="./afadjato_data/raw/post")[0]

# Real acquisition datetime for each scene -- needed for this
# pipeline's own real, per-date solar position calculation.
pre_datetime = pre_results[0].datetime
post_datetime = post_results[0].datetime
print(f"Pre-event real acquisition: {pre_datetime}")
print(f"Post-event real acquisition: {post_datetime}")

extract_dir = Path("./afadjato_data/extracted")
with zipfile.ZipFile(pre_download.output_path) as zf:
    zf.extractall(extract_dir / "pre")
with zipfile.ZipFile(post_download.output_path) as zf:
    zf.extractall(extract_dir / "post")


## Step 2 — Compute real NDVI for both dates

`client.indices.ndvi()` needs real red (B04) and NIR (B08) bands.


In [ ]:
import glob

def find_band(directory, band):
    return glob.glob(f"{directory}/**/*_{band}_10m.jp2", recursive=True)[0]

pre_red = find_band(extract_dir / "pre", "B04")
pre_nir = find_band(extract_dir / "pre", "B08")
post_red = find_band(extract_dir / "post", "B04")
post_nir = find_band(extract_dir / "post", "B08")

ndvi_pre_result = client.indices.ndvi(pre_red, pre_nir, output="./afadjato_data/ndvi_pre.tif")
ndvi_post_result = client.indices.ndvi(post_red, post_nir, output="./afadjato_data/ndvi_post.tif")


## Step 3 — Search for a real DEM

The real Copernicus DEM (30m, global, no-auth once an API key is
registered) via `opentopography` covers this real terrain.


In [ ]:
from pygeofetch.models.search_query import SearchQuery as SQ

dem_query = SQ(bbox=AOI)
dem_results = client.search(dem_query, providers=["opentopography"])
cop_dem = next(r for r in dem_results if r.properties.get("dem_type") == "COP30")
dem_download = client.download([cop_dem], destination="./afadjato_data/dem")[0]


## Step 4 — Run the real terrain-correction pipeline

Real, independent solar position for each date via the pipeline's own
Spencer (1971)-based calculation — no manual astronomy needed.


In [ ]:
from pygeofetch.multisensor import terrain_corrected_change_pipeline

# Real Afadjato summit coordinates, used as the scene-center reference
# for solar position -- close enough to the whole AOI's real latitude
# range for the cosine correction to be meaningful across it.
LATITUDE, LONGITUDE = 7.027, 0.603

result = terrain_corrected_change_pipeline(
    optical_pre_path=ndvi_pre_result.output_path,
    optical_post_path=ndvi_post_result.output_path,
    dem_path=dem_download.output_path,
    pre_datetime=pre_datetime, post_datetime=post_datetime,
    latitude=LATITUDE, longitude=LONGITUDE,
    output_dir="./afadjato_data/terrain_change",
    change_threshold=0.1,
)

assert result.success, result.error
print(f"Pre solar position: {result.metadata['solar_position_pre']}")
print(f"Post solar position: {result.metadata['solar_position_post']}")
print(f"Real pct changed (corrected): {result.metadata['pct_changed']}%")


## Step 5 — Compare against a naive, uncorrected difference

The real, honest way to see whether terrain correction actually
mattered for this specific real scene: compute the naive NDVI
difference too, and compare its flagged-change area to the corrected
result above.


In [ ]:
import numpy as np
import rasterio

with rasterio.open(ndvi_pre_result.output_path) as src:
    ndvi_pre = src.read(1)
with rasterio.open(ndvi_post_result.output_path) as src:
    ndvi_post = src.read(1)

naive_diff = ndvi_post - ndvi_pre
naive_pct_changed = 100 * (np.abs(naive_diff) > 0.1).mean()
print(f"Naive (uncorrected) pct changed: {naive_pct_changed:.1f}%")
print(f"Terrain-corrected pct changed: {result.metadata['pct_changed']}%")


## Interpretation — what to actually look for

- **If the naive and corrected percentages are close**, the real
  illumination difference between these two specific dates happened to
  be small enough not to matter much for this real terrain — a real,
  legitimate, informative outcome, not a failure of the method.
- **If the naive percentage is substantially higher**, that gap is the
  real, quantified illusion of change the terrain correction removed —
  directly matching the real, controlled test in this project's own
  test suite (a synthetic scenario with a genuinely unchanged surface
  where the naive comparison falsely flagged change purely from
  illumination).
- **Check the printed solar positions**: a January (dry season) versus
  July (wet season) pass at this real Volta Region latitude should
  show a real, meaningfully different sun position — if they look
  nearly identical, double-check the real acquisition datetimes were
  parsed correctly before trusting the rest of the result.

## Honest limitations of this specific project

- Cosine correction (the method this pipeline uses) is the simplest
  real, standard topographic correction — it may under-correct on
  Afadjato's steepest slopes or under dense canopy, where more
  sophisticated real methods (C-correction, Minnaert, SCS+C) would do
  better. This is a documented, real limitation of the method itself,
  not this specific implementation.
- `LATITUDE, LONGITUDE` use the real summit coordinates as a single
  scene-center reference — for a genuinely large AOI, solar geometry
  varies slightly across it; this is a real, reasonable approximation
  for a 5km box, not exact everywhere within it.
